### Evaluation of RAG Application & Chatbot

#### What to Evaluate?

1. **Which LLM model?**
2. **Ground Truth Output**
3. **Evaluation Metrics**

#### Evaluation Process

Gather Data Points
→ LLM as a Judge
→ Evaluation Metrics
→ Compare Multiple LLM Models

#### LLM as a Judge

- Use an LLM to evaluate the chatbot's output.
- **LangSmith** can be used for evaluation.

#### Data Points

Input → Chatbot → Output

- Input: User query
- Output: Chatbot's response
- Ground Truth: Expected/correct output

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"

In [ ]:
## Step 1: Gather data points

from langsmith import Client as LangSmithClient

langsmith_client = LangSmithClient()

## Define the test datasets
dataset_name = "Simple chatbot evaluation"
# dataset = langsmith_client.create_dataset(dataset_name)

examples = [
    {
        "input": "What is RAG?",
        "output": "RAG stands for Retrieval-Augmented Generation. It retrieves relevant information from a knowledge base and provides it to an LLM to generate a more accurate answer."
    },
    {
        "input": "What is a vector embedding?",
        "output": "A vector embedding is a numerical representation of data such as text. Similar meanings tend to have similar vectors."
    },
    {
        "input": "What is FAISS?",
        "output": "FAISS is a library for efficient similarity search of vector embeddings."
    },
    {
        "input": "What is a Transformer?",
        "output": "A Transformer is a neural network architecture that uses attention mechanisms to process relationships between tokens in a sequence."
    },
    {
        "input": "What is cosine similarity?",
        "output": "Cosine similarity measures how similar two vectors are based on the angle between them."
    }
]

inputs = [
    {"question": example["input"]}
    for example in examples
]

outputs = [
    {"answer": example["output"]}
    for example in examples
]

# langsmith_client.create_examples(
#     dataset_id="fc7920b1-58e0-48f8-972a-3e925f8983e6",
#     inputs=inputs,
#     outputs=outputs
# )

print(f"Created {len(examples)} examples")

In [ ]:
## Step 2: LLM as a judge

from openai import OpenAI
from langsmith import wrappers

groq_client = wrappers.wrap_openai(OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
    max_retries=3,
))

eval_instructions = "You're an expert professor specialized in grading students' answers. Evaluate correctness, completeness, and clarity. Reply with exactly CORRECT or INCORRECT."


def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    predicted_answer = outputs.get("response") or outputs.get("answer")
    reference_answer = reference_outputs.get("answer")
    if not predicted_answer or not reference_answer:
        return False

    user_content = f"""
Question:
{inputs['question']}

Reference answer:
{reference_answer}

Predicted answer:
{predicted_answer}

Reply with exactly CORRECT or INCORRECT.
"""

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        temperature=0,
        messages=[
            {"role": "system", "content": eval_instructions},
            {"role": "user", "content": user_content},
        ],
    )

    return response.choices[0].message.content.strip().upper() == "CORRECT"

In [ ]:
## Concision:
## The response is concise when it is less than twice the length of the reference answer.

def concision(outputs: dict, reference_outputs: dict) -> bool:
    predicted_answer = outputs.get("response") or outputs.get("answer")
    reference_answer = reference_outputs.get("answer")
    if not predicted_answer or not reference_answer:
        return False

    return len(predicted_answer) < 2 * len(reference_answer)

In [ ]:
## Run the application

default_instructions = "Respond to the user's question in a short and concise manner (one short answer)."


def my_app(
    question: str,
    model: str = "openai/gpt-oss-120b",
    instructions: str = default_instructions,
) -> str:
    response = groq_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    )
    return response.choices[0].message.content or ""


In [ ]:
## Call my_app for every data point

def ls_target(inputs: dict) -> dict:
    return {"response": my_app(inputs["question"])}

In [ ]:
## Run the evaluation
exp_results = langsmith_client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[correctness, concision],
    experiment_prefix="groq-gpt-oss-120b-chatbot",
    max_concurrency=1,
)